# FairCV synthetic headshots: preview run

This notebook generates a small balanced preview of synthetic headshots for FairCV resumes. It takes `PER_GROUP` CVs from each of the 6 gender × ethnicity groups. It then checks with CLIP that each face shows its labelled attributes and draws a contact sheet.

**Runtime:** select *Runtime → Change runtime type → T4 GPU* (free tier is fine).

| Setting | Time on a T4 (approx.) |
|---|---|
| `sdxl-turbo`, `PER_GROUP=10` (60 images) | ~3–5 min including model download |
| `sdxl`, `PER_GROUP=2` (12 images) | ~10 min |

**Private repo?** Add a GitHub token with read access as a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar).

In [ ]:
#@title Settings
BACKEND = "sdxl-turbo"  #@param ["sdxl-turbo", "sdxl"]
PER_GROUP = 10  #@param {type:"integer"}
SPLIT = "test"  #@param ["test", "train", "all"]
OCCUPATION_ATTIRE = False  #@param {type:"boolean"}
SEED = 0  #@param {type:"integer"}
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
REPO = "shubhaguha/faircv-synthetic-faces"
OUT = f"/content/out/preview_{BACKEND}"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: switch the runtime to T4 GPU"

In [ ]:
import os, subprocess
token = ""
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass
url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
if not os.path.exists("/content/faircv-synthetic-faces"):
    subprocess.run(["git", "clone", "-q", url, "/content/faircv-synthetic-faces"], check=True)
%cd /content/faircv-synthetic-faces
!pip install -q -e ".[gen]"

In [ ]:
# FairCVdb (~200 MB) is stored with Git LFS in BiDAlab/FairCVtest.
!mkdir -p data && [ -s data/FairCVdb.npy ] || curl -sSL -o data/FairCVdb.npy \
  https://media.githubusercontent.com/media/BiDAlab/FairCVtest/master/data/FairCVdb.npy
!ls -lh data/FairCVdb.npy

In [ ]:
attire = "--occupation-attire" if OCCUPATION_ATTIRE else ""
!python -m faircv_synth.generate --faircv data/FairCVdb.npy --out {OUT} \
    --backend {BACKEND} --split {SPLIT} --per-group {PER_GROUP} --seed {SEED} {attire}

In [ ]:
# CLIP zero-shot check: does each face show its labelled gender and ethnicity?
!python -m faircv_synth.verify {OUT}/manifest.csv

In [ ]:
!python -m faircv_synth.audit {OUT}/manifest.csv | head -20

In [ ]:
# One row per group; images that failed the CLIP check have a red border.
!python -m faircv_synth.contact_sheet {OUT}/manifest.csv --per-row {min(PER_GROUP, 10)}
from IPython.display import Image as Show, display
display(Show(f"{OUT}/contact_sheet.png"))

In [ ]:
import pandas as pd
fid = pd.read_csv(f"{OUT}/fidelity.csv")
failed = fid[~fid.passes]
print(f"{len(failed)} failures")
from PIL import Image
for row in failed.head(12).itertuples():
    print(row.cv_id, "label:", row.gender, row.ethnicity, "| CLIP:", row.pred_gender, row.pred_ethnicity)
    display(Image.open(f"{OUT}/{row.image_path}").resize((256, 256)))

In [ ]:
# Download the results as a zip, or copy them to Google Drive.
import shutil
archive = shutil.make_archive(OUT, "zip", OUT)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy(archive, "/content/drive/MyDrive/")
    print("copied to MyDrive/", os.path.basename(archive))
else:
    from google.colab import files
    files.download(archive)